# Qwen-14B Map Reasoning Probe

This notebook tests whether `Qwen/Qwen2.5-14B-Instruct` can read different LPLH2 world-map representations and answer route questions.

It does not run Zork. It only loads Qwen-14B and asks map-comprehension questions against a static discovered Zork1 map.

The test now runs each question in three modes:
- **current_kg_format**: the model receives the current LPLH2 KG prompt format: `map -> room -> temp_have/have/direction/may_direction`.
- **raw_graph**: the model receives the proposed cleaner `navigation_graph` JSON and must compute the route itself.
- **assisted_routes**: the notebook computes BFS route hints from the question's start room, then the model may use those hints.

This lets us compare the current KG format against a cleaner graph and against deterministic route assistance.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece


In [ ]:
import copy, json, re, textwrap, collections
from collections import deque
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
MAX_NEW_TOKENS = 256

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("CUDA available:", torch.cuda.is_available())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Loaded", MODEL_NAME, "with dtype", dtype)


## Static World Model

This uses the simpler JSON shape we discussed:
- current location,
- current room state,
- navigation graph,
- optional code-computed routes from the current location,
- frontier.

I added `known_object_locations` only for object-route questions. If an object is not in that field, the model should say it is unknown rather than using outside Zork knowledge.


In [ ]:
BASE_WORLD_MODEL = {
    "current_location": "Living Room",
    "current_room_state": {
        "inventory": ["sword", "lantern", "sack", "bottle"],
        "visible_objects": ["trophy case", "rug", "trap door"],
        "object_states": {
            "trophy case": "open",
            "rug": "moved",
            "trap door": "revealed, open"
        },
        "confirmed_exits": {
            "east": "Kitchen",
            "down": "Cellar"
        },
        "blocked_exits": {
            "west": "nailed wooden door"
        },
        "untried_exits": ["north", "south"]
    },
    "navigation_graph": {
        "West of House": {"north": "North of House"},
        "North of House": {"south": "West of House", "north": "Forest Path", "east": "Behind House"},
        "Forest Path": {"south": "North of House", "north": "Clearing", "up": "Up a Tree"},
        "Up a Tree": {"down": "Forest Path"},
        "Clearing": {"south": "Forest Path"},
        "Behind House": {"west": "North of House", "east": "Kitchen"},
        "Kitchen": {"west": "Living Room", "east": "Behind House", "up": "Attic"},
        "Attic": {"down": "Kitchen"},
        "Living Room": {"east": "Kitchen", "down": "Cellar"},
        "Cellar": {"up": "Living Room", "north": "Troll Room", "south": "East-West Passage"},
        "Troll Room": {"south": "Cellar", "east": "East-West Passage"},
        "East-West Passage": {"west": "Troll Room", "east": "Loud Room", "north": "Round Room"},
        "Loud Room": {"west": "East-West Passage"},
        "Round Room": {"south": "East-West Passage"}
    },
    "known_object_locations": {
        "egg": "Up a Tree",
        "lantern": "Living Room",
        "sword": "Living Room",
        "sack": "Kitchen",
        "bottle": "Kitchen",
        "axe": "Troll Room"
    },
    "routes_from_current_location": {},
    "frontier": {
        "Clearing": ["east", "west", "north"],
        "Cellar": ["west", "down"],
        "East-West Passage": ["down"],
        "Loud Room": ["up"]
    }
}

ROOM_OBJECTS = {
    "West of House": ["mailbox"],
    "North of House": ["white house", "path"],
    "Forest Path": ["tree"],
    "Up a Tree": ["bird nest", "egg"],
    "Clearing": ["pile of leaves"],
    "Behind House": ["window"],
    "Kitchen": ["sack", "bottle", "table"],
    "Attic": [],
    "Living Room": ["trophy case", "rug", "trap door"],
    "Cellar": [],
    "Troll Room": ["axe"],
    "East-West Passage": [],
    "Loud Room": [],
    "Round Room": [],
}

STANDARD_DIRECTIONS = [
    "north", "south", "east", "west",
    "northeast", "northwest", "southeast", "southwest",
    "up", "down",
]

ROOM_LOCAL_STATE = {
    "Living Room": {
        "visible_objects": ["trophy case", "rug", "trap door"],
        "object_states": {"trophy case": "open", "rug": "moved", "trap door": "revealed, open"},
        "blocked_exits": {"west": "nailed wooden door"},
        "untried_exits": ["north", "south"],
    },
    "Kitchen": {
        "visible_objects": ["sack", "bottle", "table"],
        "object_states": {},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Forest Path": {
        "visible_objects": ["tree"],
        "object_states": {},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Up a Tree": {
        "visible_objects": ["bird nest", "egg"],
        "object_states": {},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Troll Room": {
        "visible_objects": ["axe"],
        "object_states": {"troll": "defeated or absent"},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Loud Room": {
        "visible_objects": [],
        "object_states": {"condition": "echo may distort commands"},
        "blocked_exits": {},
        "untried_exits": ["up"],
    },
}

def shortest_route(graph, start, target):
    if start not in graph or target not in graph:
        return None
    q = deque([(start, [], [start])])
    seen = {start}
    while q:
        room, route, path = q.popleft()
        if room == target:
            return {"route": route, "path": path}
        for direction, nxt in graph.get(room, {}).items():
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, route + [direction], path + [nxt]))
    return None

def build_world_model_for_case(case, include_route_hints):
    world = copy.deepcopy(BASE_WORLD_MODEL)
    start = case["start"]
    graph = world["navigation_graph"]
    local = ROOM_LOCAL_STATE.get(start, {"visible_objects": [], "object_states": {}, "blocked_exits": {}, "untried_exits": []})
    world["current_location"] = start
    world["current_room_state"] = {
        "inventory": BASE_WORLD_MODEL["current_room_state"]["inventory"],
        "visible_objects": local["visible_objects"],
        "object_states": local["object_states"],
        "confirmed_exits": graph.get(start, {}),
        "blocked_exits": local["blocked_exits"],
        "untried_exits": local["untried_exits"],
    }
    world["routes_from_current_location"] = {}
    if include_route_hints:
        for target in graph:
            route = shortest_route(graph, start, target)
            if route and target != start:
                world["routes_from_current_location"][target] = route["route"]
    return world


def build_current_kg_format_for_case(case):
    """Render the same static world in the current KGMap.to_prompt_string shape."""
    start = case["start"]
    graph = BASE_WORLD_MODEL["navigation_graph"]
    map_nodes = {}
    for loc in graph:
        # In live KGMap, may_direction means possible but unconfirmed directions.
        # Here we use the static frontier when known; otherwise keep it empty so this
        # test focuses on format rather than a guessed exploration history.
        may_direction = BASE_WORLD_MODEL["frontier"].get(loc, [])
        map_nodes[loc] = {
            "temp_have": BASE_WORLD_MODEL["current_room_state"]["inventory"] if loc == start else [],
            "have": ROOM_OBJECTS.get(loc, []),
            "may_have": [],
            "direction": graph.get(loc, {}),
            "may_direction": may_direction,
        }
        local = ROOM_LOCAL_STATE.get(loc)
        if local and local.get("object_states"):
            map_nodes[loc]["relations"] = [
                {"subject": name, "relation": "state", "object": state}
                for name, state in local["object_states"].items()
            ]
        if local and local.get("blocked_exits"):
            map_nodes[loc]["needs"] = [
                f"{direction} blocked: {reason}"
                for direction, reason in local["blocked_exits"].items()
            ]
    return {
        "current_location": start,
        "visited_rooms": list(graph.keys()),
        "map": map_nodes,
    }


print(json.dumps(build_world_model_for_case({"start": "Living Room"}, include_route_hints=True), indent=2))


In [ ]:
TEST_CASES = [
    {
        "id": "route_living_to_troll",
        "question": "You are in Living Room. To go from Living Room to Troll Room, what route should you follow?",
        "start": "Living Room",
        "target_location": "Troll Room",
    },
    {
        "id": "route_loud_to_kitchen",
        "question": "You are in Loud Room now. What route gets you back to Kitchen?",
        "start": "Loud Room",
        "target_location": "Kitchen",
    },
    {
        "id": "route_clearing_to_living",
        "question": "You are in Clearing. What route should you follow to reach Living Room?",
        "start": "Clearing",
        "target_location": "Living Room",
    },
    {
        "id": "object_egg_from_living",
        "question": "You are in Living Room and need the egg. Where is the egg, and what route should you follow to get there?",
        "start": "Living Room",
        "target_object": "egg",
    },
    {
        "id": "object_axe_from_kitchen",
        "question": "You are in Kitchen and need the axe. Where is the axe, and what route should you follow to get there?",
        "start": "Kitchen",
        "target_object": "axe",
    },
    {
        "id": "unknown_route_to_dam",
        "question": "You are in Living Room. What route should you follow to reach Dam?",
        "start": "Living Room",
        "target_location": "Dam",
    },
    {
        "id": "unknown_object_rope",
        "question": "You are in Living Room and need the rope. Where is the rope, and what route should you follow?",
        "start": "Living Room",
        "target_object": "rope",
    },
]

def expected_for_case(case):
    graph = BASE_WORLD_MODEL["navigation_graph"]
    start = case["start"]
    if "target_object" in case:
        obj = case["target_object"]
        loc = BASE_WORLD_MODEL["known_object_locations"].get(obj)
        if not loc:
            return {"known": False, "target_object": obj, "target_location": None, "route": [], "path": []}
        route = shortest_route(graph, start, loc)
        return {"known": bool(route), "target_object": obj, "target_location": loc, **(route or {"route": [], "path": []})}
    target = case["target_location"]
    route = shortest_route(graph, start, target)
    return {"known": bool(route), "target_location": target, **(route or {"route": [], "path": []})}

print("Expected answers:")
for case in TEST_CASES:
    expected = expected_for_case(case)
    print("\n" + case["id"])
    print("Q:", case["question"])
    print("known:", expected["known"])
    print("target_location:", expected.get("target_location"))
    if expected.get("target_object"):
        print("target_object:", expected.get("target_object"))
    print("route:", expected.get("route"))
    print("path:", expected.get("path"))


In [ ]:
SYSTEM_PROMPT = """You are a map-comprehension evaluator for a text adventure agent.
Use ONLY the provided JSON world model. Do not use outside knowledge about Zork.
If a requested room, object, or route is not present or not connected in the JSON, say it is unknown.
Return JSON only between |start| and |end|.

Required output schema:
{
  "known": true or false,
  "start": "room name",
  "target_location": "room name or null",
  "target_object": "object name or null",
  "route": ["direction", "direction"],
  "path": ["room", "room"],
  "reason": "one short sentence"
}

Rules:
- route is a list of direction commands only, such as ["down", "north"].
- path is a list of room names from start to target.
- If routes_from_current_location contains the target room, prefer that route exactly.
- If navigation_graph exists, confirmed exits are in navigation_graph[room].
- If map exists, this is the current KG format: confirmed exits are in map[room]["direction"], room objects are in map[room]["have"], inventory is in map[current_location]["temp_have"], and may_direction is only unconfirmed possible exits.
- For object questions, use known_object_locations if present; otherwise infer the object's room from map[room]["have"].
- If unknown, set known=false, route=[], path=[], and explain what is missing from the JSON.
"""

TEST_MODES = [
    {"id": "current_kg_format", "format": "kg", "include_route_hints": False},
    {"id": "raw_graph", "format": "clean", "include_route_hints": False},
    {"id": "assisted_routes", "format": "clean", "include_route_hints": True},
]

def build_user_prompt(case, world):
    return f"""World model JSON:
{json.dumps(world, indent=2)}

Question:
{case['question']}
"""

def ask_qwen(case, world):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(case, world)},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def parse_answer(raw):
    m = re.search(r"\|start\|\s*(.*?)\s*\|end\|", raw, re.S)
    body = m.group(1).strip() if m else raw.strip()
    try:
        return json.loads(body)
    except Exception as e:
        return {"parse_error": str(e), "raw_body": body}

def normalize_route(route):
    if not isinstance(route, list):
        return None
    return [str(direction).strip().lower() for direction in route]

def follow_route(graph, start, route):
    room = start
    path = [room]
    for direction in route:
        exits = graph.get(room, {})
        if direction not in exits:
            return None
        room = exits[direction]
        path.append(room)
    return {"target_location": room, "path": path}

def route_matches(answer, expected, case):
    if "parse_error" in answer:
        return False
    if bool(answer.get("known")) != bool(expected.get("known")):
        return False
    if not expected.get("known"):
        return True
    route = normalize_route(answer.get("route"))
    if route is None:
        return False
    walked = follow_route(BASE_WORLD_MODEL["navigation_graph"], case["start"], route)
    if not walked:
        return False
    return walked["target_location"] == expected.get("target_location")

ALL_RESULTS = []
for mode in TEST_MODES:
    mode_results = []
    print("\n" + "#" * 100)
    print("MODE:", mode["id"], "| include_route_hints=", mode["include_route_hints"])
    for case in TEST_CASES:
        if mode["format"] == "kg":
            world = build_current_kg_format_for_case(case)
        else:
            world = build_world_model_for_case(case, mode["include_route_hints"])
        print("\n" + "=" * 90)
        print(case["id"])
        print(case["question"])
        if mode["include_route_hints"]:
            print("ROUTE HINTS FROM START:", world["routes_from_current_location"])
        elif mode["format"] == "kg":
            print("CURRENT KG ROOM:", json.dumps(world["map"].get(case["start"], {}), indent=2))
        expected = expected_for_case(case)
        print("EXPECTED:", expected)
        raw = ask_qwen(case, world)
        parsed = parse_answer(raw)
        ok = route_matches(parsed, expected, case)
        print("RAW:\n", raw)
        print("PARSED:", parsed)
        print("PASS:", ok)
        row = {"mode": mode["id"], "case": case, "expected": expected, "raw": raw, "parsed": parsed, "pass": ok}
        mode_results.append(row)
        ALL_RESULTS.append(row)
    print("\nSUMMARY FOR", mode["id"])
    print(sum(1 for r in mode_results if r["pass"]), "/", len(mode_results), "passed")

print("\nCOMPARISON")
for mode in TEST_MODES:
    rows = [r for r in ALL_RESULTS if r["mode"] == mode["id"]]
    print(mode["id"], sum(1 for r in rows if r["pass"]), "/", len(rows))
